In [1]:
from pathlib import Path

import pandas as pd

from src.plots import generate_data_plot

DATA_FILE = Path("test_data.xlsx")

In [2]:
df = pd.read_excel(DATA_FILE)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   datetime                          48 non-null     datetime64[ns]
 1   pv production, kWh                48 non-null     float64       
 2   electrical consumption, kWh       48 non-null     float64       
 3   lcos, c/kWh                       48 non-null     float64       
 4   electricity selling price, c/kWh  48 non-null     float64       
 5   electricity buying price c/kWh    48 non-null     float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 2.4 KB


In [4]:
df.head()

,datetime,"pv production, kWh","electrical consumption, kWh","lcos, c/kWh","electricity selling price, c/kWh",electricity buying price c/kWh
0,2023-04-05 00:00:00,0.0,69.540000,9.3,11.826,24.826
1,2023-04-05 01:00:00,0.0,74.941647,9.3,11.961,24.961
2,2023-04-05 02:00:00,0.0,83.941647,9.3,11.825,24.825
3,2023-04-05 03:00:00,0.0,114.641647,9.3,12.499,25.499
4,2023-04-05 04:00:00,0.0,281.117334,9.3,13.446,26.446


Taking a glimpse at the data. Here it makes sense to group it by unit (kWh vs c/kWh) and generate a twinx plot.

(Double click on the legend to isolate.)

In [5]:
generate_data_plot(df).show()


## Optimization with `pyomo`

In [6]:
from src.pyomo_setup import EnergyFlowOptimization
from src.plots import generate_flows_plot

In [7]:
efo = EnergyFlowOptimization(df)

In [8]:
efo.solve()

GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --write C:\Users\Airat\AppData\Local\Temp\tmpydqxxms9.glpk.raw --wglp C:\Users\Airat\AppData\Local\Temp\tmpmf5hdlt3.glpk.glp
 --cpxlp C:\Users\Airat\AppData\Local\Temp\tmpfwh9lxvi.pyomo.lp
Reading problem data from 'C:\Users\Airat\AppData\Local\Temp\tmpfwh9lxvi.pyomo.lp'...
576 rows, 576 columns, 1295 non-zeros
3752 lines were read
Writing problem data to 'C:\Users\Airat\AppData\Local\Temp\tmpmf5hdlt3.glpk.glp'...
2909 lines were written
GLPK Simplex Optimizer, v4.65
576 rows, 576 columns, 1295 non-zeros
Preprocessing...
294 rows, 395 columns, 790 non-zeros
Scaling...
 A: min|aij| =  9.200e-01  max|aij| =  1.000e+00  ratio =  1.087e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 294
*     0: obj =   2.176866584e+05 inf =   0.000e+00 (118)
*   120: obj =   1.425887292e+05 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 secs
Memory used: 0.6 Mb 

In [9]:
generate_flows_plot(efo).show()

In [10]:
costs_table = efo.generate_costs_table()
costs_table

,Metric (cents),Value
0,Total Cost,142588.729195
1,Total Buy Cost,161147.525444
2,Total Sell Revenue,22464.796249
3,Battery Discharge Cost,3906.000000
